<a href="https://colab.research.google.com/github/ahmed-59/BigBountyRecon/blob/main/section2/section2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 2 Assignment: Is this car expensive? 🚗💰

In Section 2 we turned the churn data into numbers with **one-hot encoding**, trained a **logistic regression** model, measured it, and **saved** it. In this assignment you do the **same steps on the car data** from the Section 1 assignment.

**The story:** you still work at *AutoSell*. The website wants to show a **"Premium" badge** on expensive listings. When a seller adds a car, a model should guess: *is this car more expensive than the typical car?* That is the `is_expensive` column you created last time (1 = above the median price, 0 = not).

So this is a **classification** problem, exactly like churn:

| Section 2 (churn) | This assignment (cars) |
|---|---|
| a customer | a car |
| `churn` (1 = leaves) | `is_expensive` (1 = above the median price) |
| `contract=month-to-month` | `make=bmw`, `vehicle_style=convertible`, ... |

**Your data is unique.** The setup cell gives you your own random sample of the cars, chosen by your college ID. Your numbers will be different from your friends' numbers, so copying answers won't work. Helping each other understand the steps is fine. 🙂

## Before you start (read this!)

1. **File → Save a copy in Drive.** If you skip this, your work is not saved.
2. **Write your college ID** in the first code cell below, then run it.
3. **Go step by step.** Every step explains what to do. Write your code where you see `...`, then run the `grader.check(...)` cell under it.
4. **About `grader.check`:** ✅ means your answer has the **right form** (for example, a number where we expect a number). It does **not** mean the answer is correct. Your real grade is calculated after the deadline.
5. **Rules:**
    - Don't rename the variables we ask for (`X_train`, `model`, `accuracy`, ...). The grader looks for these exact names.
    - Don't change or delete the setup cell, the "already done for you" cells, or the `grader.check` cells.
    - Use only what the steps ask for. You don't need any extra libraries.
6. **How to submit** is explained at the end of the notebook.

In [23]:
# write your college ID between the quotes, for example: "202202"
STUDENT_ID = "234849"

# ---- setup: do not change anything below this line ----
if not isinstance(STUDENT_ID, str) or not STUDENT_ID.isdigit():
    raise ValueError('Write your college ID on the first line, between quotes, for example: "202202"')
import os
if not os.path.exists("car_prices.csv"):   # first run on Colab: get the data and the checks
    !git clone -q --depth 1 https://github.com/omaramir2001/ml-assignments.git
    !cp -r ml-assignments/section2/. .
    %pip install -q otter-grader
import otter
import numpy as np
import pandas as pd
grader = otter.Notebook()
df = pd.read_csv("car_prices.csv").sample(n=8000 + int(STUDENT_ID) % 3000, random_state=int(STUDENT_ID)).reset_index(drop=True)
print(f"Your data is ready: {len(df)} cars.")

Your data is ready: 8849 cars.


# Part 0: Section 1 recap (already done for you)

The next cell repeats the data preparation from the Section 1 assignment: clean names and values, fill the missing values, create `is_expensive`, and split into train and validation. **Just run it.** Read the comments if you want to remember why each line is there.

In [24]:
from sklearn.model_selection import train_test_split

# 1. uniform column names and values
df.columns = df.columns.str.lower().str.replace(' ', '_')
text_columns = list(df.select_dtypes(exclude='number').columns)
df[text_columns] = df[text_columns].apply(lambda s: s.str.lower().str.replace(' ', '_'))

# 2. fill the missing values (same decisions as in Section 1)
df['engine_cylinders'] = df['engine_cylinders'].fillna(0)            # electric cars have no cylinders
df['engine_hp'] = df['engine_hp'].fillna(df['engine_hp'].median())
df['market_category'] = df['market_category'].fillna('unknown')
df['engine_fuel_type'] = df['engine_fuel_type'].fillna(df['engine_fuel_type'].mode()[0])
df['number_of_doors'] = df['number_of_doors'].fillna(df['number_of_doors'].mode()[0])

# 3. the target: 1 if the car costs more than the median price
df['log_msrp'] = np.log1p(df['msrp'])
df['is_expensive'] = (df['msrp'] > df['msrp'].median()).astype(int)

# 4. split: 80% train, 20% validation
df_train, df_val = train_test_split(df, test_size=0.2, random_state=42)
print(len(df_train), len(df_val))

7079 1770


These are the columns the model is allowed to use. Like in the section, we keep two lists.

🔎 Notice what is **not** in the lists:
- `msrp` and `log_msrp`: they *are* the price. If the model could see the price, guessing "is it expensive?" would be cheating. This mistake is called **data leakage**.
- `is_expensive`: that's the answer itself.
- `model`: there are almost 1,000 different car models, which is too many columns for now.

`number_of_doors` is a number (2, 3 or 4), so it goes with the numerical columns.

In [25]:
categorical = ['make', 'engine_fuel_type', 'transmission_type', 'driven_wheels',
               'market_category', 'vehicle_size', 'vehicle_style']
numerical = ['year', 'engine_hp', 'engine_cylinders', 'highway_mpg',
             'city_mpg', 'popularity', 'number_of_doors']

# Part 1: One-hot encoding

## Step 1: The answers (y)

The model learns from examples **and** their answers. Like `y_train = df_train.churn.values` in the section, take the `is_expensive` column from each part:

- `y_train`: the answers of the training cars
- `y_val`: the answers of the validation cars

Use `.values` so you get NumPy arrays.

In [26]:
y_train = df_train['is_expensive'].values
y_val = df_val['is_expensive'].values

In [27]:
grader.check("q1")

q1 results: All test cases passed!

## Step 2: From a table to a list of dictionaries

`DictVectorizer` wants **one dictionary per car**. In the section we did this with `to_dict(orient='records')`.

Take only the `categorical + numerical` columns from `df_train`, and turn them into a list of dictionaries called `train_dicts`. Do the same for `df_val` and call it `val_dicts`.

⚠️ Don't forget the `[categorical + numerical]` part. Without it, the dictionaries would include `msrp`, and the model would see the price (data leakage!).

In [28]:
train_dicts = df_train[categorical + numerical].to_dict(orient='records')
val_dicts = df_val[categorical + numerical].to_dict(orient='records')

In [29]:
grader.check("q2")

q2 results: All test cases passed!

## Step 3: One-hot encoding with DictVectorizer

Now turn the dictionaries into a matrix of numbers, like in the section:

1. Create a `DictVectorizer` called `dv` with `sparse=False`.
2. **Fit and transform** on `train_dicts` → `X_train`.
3. **Only transform** on `val_dicts` → `X_val`.
4. Store the number of features (columns) in `n_features`. Hint: `dv.get_feature_names_out()` gives all the feature names, and `len(...)` counts them.

🔎 Remember from the section: we `fit` only on the training data. The validation data plays the role of *new cars the model has never seen*, so we must not learn anything from it, not even the list of categories.

In [30]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
n_features = len(dv.get_feature_names_out())

print(X_train.shape, X_val.shape, n_features)

(7079, 160) (1770, 160) 160


In [31]:
grader.check("q3")

q3 results: All test cases passed!

# Part 2: Logistic regression

## Step 4: The sigmoid function

Logistic regression computes a score, then squeezes it between 0 and 1 with the **sigmoid** function:

$$\text{sigmoid}(z) = \frac{1}{1 + e^{-z}}$$

Complete the function `sigmoid(z)` below, like in the section. Use `np.exp`, so it works for one number and for a whole array.

Check yourself: `sigmoid(0)` should be `0.5`, a big positive number should give almost 1, and a big negative number almost 0.

In [32]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

sigmoid(np.array([-7, -1, 0, 1, 7])).round(3)

array([0.001, 0.269, 0.5  , 0.731, 0.999])

In [33]:
grader.check("q4")

q4 results: All test cases passed!

## Step 5: Train the model

Create a `LogisticRegression` called `model` with `solver='liblinear'` (like the section), and **fit** it on `X_train` and `y_train`.

Then store the bias (the intercept, $w_0$) in a variable called `bias`. Hint: `model.intercept_` is an array with one value, so take `model.intercept_[0]`.

In [34]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(solver='liblinear')

model.fit(X_train, y_train)
bias = model.intercept_[0]

bias

np.float64(-0.4287393603140779)

In [35]:
grader.check("q5")

q5 results: All test cases passed!

## Step 6: Read the weights

Every feature has a weight in `model.coef_[0]`. Like in the section, put the weights next to the feature names, so you can read them:

```python
weights = pd.Series(model.coef_[0], index=dv.get_feature_names_out())
```

Then answer:
1. `bmw_weight`: the weight of the feature `'make=bmw'`.
2. `top_positive_feature`: the **name** of the feature with the **biggest** weight (it pushes a car the most toward "expensive").
3. `top_negative_feature`: the **name** of the feature with the **smallest** (most negative) weight.

Hint: `weights['make=bmw']`, `weights.idxmax()`, `weights.idxmin()`.

In [36]:
weights = pd.Series(model.coef_[0], index=dv.get_feature_names_out())

bmw_weight = weights['make=bmw']
top_positive_feature = weights.idxmax()
top_negative_feature = weights.idxmin()

print(bmw_weight, top_positive_feature, top_negative_feature)

1.081260929652347 vehicle_style=passenger_van make=suzuki


In [37]:
grader.check("q6")

q6 results: All test cases passed!

🔎 Look at your top features. Do they make sense? A positive weight means "this makes the car more likely to be expensive", and a negative weight means the opposite. This is how logistic regression stays easy to explain.

## Step 7: One prediction by hand

Before letting scikit-learn predict, let's see what the model does for **one car**, the first car in the validation set (`X_val[0]`):

1. `first_car_score`: the bias plus each feature times its weight. With NumPy that's `bias + X_val[0].dot(model.coef_[0])`.
2. `first_car_proba`: pass the score through your `sigmoid` function.

The result is the probability that this car is expensive. In the next step, scikit-learn will give you exactly the same number.

In [38]:
first_car_score = bias + X_val[0].dot(model.coef_[0])
first_car_proba = sigmoid(first_car_score)

print(first_car_score, first_car_proba)

-1.6651703549094248 0.1590691586684522


In [39]:
grader.check("q7")

q7 results: All test cases passed!

# Part 3: Predictions and accuracy

## Step 8: Predict for all the validation cars

Like in the section:

1. `y_pred`: the **soft** predictions, the probability of "expensive" for every validation car. Use `model.predict_proba(X_val)` and keep only the second column with `[:, 1]`.
2. `expensive_decision`: the **hard** predictions, `True` when `y_pred` is `0.5` or more.
3. `n_flagged`: how many validation cars get the "Premium" badge (how many `True` values). Hint: `.sum()`.

Check: `y_pred[0]` should be the same as your `first_car_proba`.

In [40]:
y_pred = model.predict_proba(X_val)[:, 1]
expensive_decision = y_pred >= 0.5
n_flagged = expensive_decision.sum()

print(y_pred[:5].round(3), n_flagged)

[0.159 0.    0.996 0.883 0.   ] 926


In [41]:
grader.check("q8")

q8 results: All test cases passed!

## Step 9: Accuracy

Accuracy is the share of cars where the model's decision matches the real answer. Compare `expensive_decision` with `y_val` and take the mean, like in the section:

`accuracy = (expensive_decision == y_val).mean()`

In [42]:
accuracy = (expensive_decision == y_val).mean()
accuracy

np.float64(0.9282485875706215)

In [43]:
grader.check("q9")

q9 results: All test cases passed!

## Step 10: The confusion matrix

Accuracy is one number. The confusion matrix shows **which** mistakes the model makes. Compute it with `confusion_matrix(y_val, expensive_decision)` and store the four counts:

|  | predicted 0 | predicted 1 |
|---|---|---|
| **actual 0** | `tn` (true negatives) | `fp` (false positives) |
| **actual 1** | `fn` (false negatives) | `tp` (true positives) |

Hint: `cm.ravel()` gives the four numbers in this order: `tn, fp, fn, tp`.

In [44]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, expensive_decision)
tn, fp, fn, tp = cm.ravel()

print(cm)

[[794  77]
 [ 50 849]]


In [45]:
grader.check("q10")

q10 results: All test cases passed!

🔎 **Think about it (not graded):** `fp` are cheap cars that got the "Premium" badge, and `fn` are expensive cars that didn't get it. Which mistake do you think annoys AutoSell's buyers more? Check: `(tp + tn) / (tn + fp + fn + tp)` gives the same number as your accuracy.

# Part 4: Pipeline and saving the model

## Step 11: Put everything in one pipeline

In the section, we used `make_pipeline` so the `DictVectorizer` and the model travel together as one object. Do the same:

1. Create `pipeline` with `make_pipeline(DictVectorizer(sparse=False), LogisticRegression(solver='liblinear'))`.
2. Fit it on `train_dicts` and `y_train` (the dictionaries, not `X_train`! The pipeline does the encoding itself).
3. Predict on `val_dicts` and compute `pipeline_accuracy`, the same way as Step 9.

You should get the same accuracy as before.

In [46]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    DictVectorizer(sparse=False),
    LogisticRegression(solver='liblinear')
)

pipeline.fit(train_dicts, y_train)

pipeline_pred = pipeline.predict(val_dicts)
pipeline_accuracy = (pipeline_pred == y_val).mean()

pipeline_accuracy

np.float64(0.9282485875706215)

In [47]:
grader.check("q11")

q11 results: All test cases passed!

## Step 12: Save the model, load it, and use it

A model only lives while the notebook runs. To use it later (for example, from the website), we save it to a file with `pickle`, like in the section.

1. Save `pipeline` to a file called `model.bin` (open the file with `'wb'`: **w**rite **b**inary).
2. Load it back into a new variable called `loaded_model` (open with `'rb'`: **r**ead **b**inary).
3. Use `loaded_model` to predict the probability for the first validation car, `val_dicts[0]`, and store it in `loaded_first_proba`. Remember: `predict_proba` wants a **list** of cars, so pass `[val_dicts[0]]`, then take `[0, 1]`.

It should be the same number as your `first_car_proba`. 🎉

In [48]:
import pickle

# 1. save the pipeline to model.bin
with open('model.bin', 'wb') as f_out:
    pickle.dump(pipeline, f_out)

# 2. load it back into loaded_model
with open('model.bin', 'rb') as f_in:
    loaded_model = pickle.load(f_in)

# 3. predict for the first validation car
loaded_first_proba = loaded_model.predict_proba([val_dicts[0]])[0, 1]

loaded_first_proba

np.float64(0.1590691586684522)

In [49]:
grader.check("q12")

q12 results: All test cases passed!

# How to submit ✅

1. **Runtime → Restart session and run all.** Wait until every cell has run, and make sure there are **no red errors**. After the deadline, the grader runs your notebook from top to bottom in exactly this way, so an error stops every answer after it.
2. **File → Download → Download .ipynb.**
3. **Rename the file to your college ID**, for example `202202.ipynb`. Use the same ID you wrote in the first cell. If they don't match, your submission is flagged.
4. **Upload it in the Google Form** your instructor shared, before the deadline.

Good to know:
- Upload the `.ipynb` file itself, not a link and not a `.py` file.
- You don't need to upload `model.bin`. The grader creates it again when it runs your notebook.
- Your grade is calculated after the deadline, from the variables you created in each step.